# RiskBrain — Colab runbook (Phase 0 setup + verification, Phase 1)

Run cells top to bottom. This notebook:
1. Checks the runtime (Python version, GPU).
2. Mounts Drive and points the HuggingFace cache there so the ~8GB Qwen3-4B
   weights survive a session disconnect.
3. Clones the repo into Drive (so your working tree also survives disconnect)
   and installs pinned dependencies, resolving the Colab-preinstalled
   torch/transformers conflict.
4. Verifies Phase 0 (loads Qwen3-4B-Instruct-2507 in 4-bit via vLLM, runs one
   generation, measures latency — labeled explicitly as a T4 functional
   check, not a representative inline-path number).
5. Runs Phase 1 (synthetic tabular fraud data, XGBoost training, fusion +
   calibration, reliability diagram) — pure CPU work, runs fine here.
6. Shows how to commit results back to GitHub so nothing is lost on
   disconnect.

Repo: https://github.com/tejasp0008/Riskbrain


## 1. Runtime check

In [ ]:
import sys
import subprocess

print(f"Python version: {sys.version}")

try:
    gpu_name = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]
    ).decode().strip()
    print(f"GPU: {gpu_name}")
except Exception as e:
    print(f"No GPU detected by nvidia-smi ({e}). "
          "In Colab: Runtime -> Change runtime type -> GPU (T4).")

py_minor = sys.version_info[:2]
if py_minor >= (3, 12):
    xgb_version = "3.4.1"   # matches requirements.txt pin as-is
elif py_minor >= (3, 10):
    xgb_version = "3.2.0"   # newest xgboost supporting Python 3.10/3.11
else:
    xgb_version = "2.1.4"   # newest xgboost supporting Python 3.9 and below

print(f"\nDetected Python {py_minor[0]}.{py_minor[1]} -> will pin xgboost=={xgb_version}")
if xgb_version != "3.4.1":
    print("NOTE: this overrides the requirements.txt pin (3.4.1, which needs "
          "Python >=3.12) for this Colab session only. The override is applied "
          "in the install cell below, not written back to requirements.txt.")


## 2. Mount Drive and set the HuggingFace cache path

This makes the downloaded model weights persist across Colab sessions —
without this, every disconnect means re-downloading ~8GB.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
HF_CACHE_DIR = "/content/drive/MyDrive/riskbrain_cache/huggingface"
ARTIFACTS_DIR = "/content/drive/MyDrive/riskbrain_cache/artifacts"
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["RISKBRAIN_ARTIFACTS_DIR"] = ARTIFACTS_DIR

print(f"HF_HOME set to {HF_CACHE_DIR}")
print(f"RISKBRAIN_ARTIFACTS_DIR set to {ARTIFACTS_DIR}")


## 3. Clone the repo into Drive

Cloning into Drive (not `/content`) means your working tree, including any
uncommitted edits, also survives a disconnect. If the repo is private, set a
`GITHUB_TOKEN` secret first (key icon in the left sidebar) and use the same
token form shown in the commit-back section below, substituted into the
clone URL.


In [ ]:
import os
import subprocess

REPO_DIR = "/content/drive/MyDrive/Riskbrain"

if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/tejasp0008/Riskbrain.git", REPO_DIR],
        check=True,
    )
else:
    print(f"{REPO_DIR} already exists, skipping clone. Run `git pull` inside it if needed.")

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())


## 4. Install pinned dependencies

Colab ships a preinstalled torch (and sometimes transformers) that can
conflict with the pinned versions in `requirements.txt`. We uninstall the
preinstalled torch stack first and let vLLM/transformers pull in the
versions they actually need, then override xgboost to the version selected
above if it differs from the requirements.txt pin.

**If this cell reports dependency conflicts or you see import errors in the
next cell, restart the runtime (Runtime -> Restart session) and re-run from
the top — this is normal after replacing torch in Colab.**


In [ ]:
import os
import subprocess
import sys

REPO_DIR = "/content/drive/MyDrive/Riskbrain"
if os.getcwd() != REPO_DIR:
    os.chdir(REPO_DIR)
    print(f"Working directory was not {REPO_DIR}, changed to it "
          "(this cell is self-contained even after a runtime restart).")

if "xgb_version" not in globals():
    py_minor = sys.version_info[:2]
    if py_minor >= (3, 12):
        xgb_version = "3.4.1"
    elif py_minor >= (3, 10):
        xgb_version = "3.2.0"
    else:
        xgb_version = "2.1.4"
    print(f"xgb_version was not set (cell 1 skipped this session), "
          f"derived it from Python {py_minor[0]}.{py_minor[1]}: {xgb_version}")

def pip(*args):
    result = subprocess.run(
        [sys.executable, "-m", "pip"] + list(args),
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(result.stdout[-3000:])
        print(result.stderr[-3000:])
        raise RuntimeError(f"pip {args[0]} failed, see output above")
    return result

pip("uninstall", "-y", "-q", "torch", "torchvision", "torchaudio")
pip("install", "-q", "-r", "requirements.txt")

if xgb_version != "3.4.1":
    pip("install", "-q", f"xgboost=={xgb_version}", "--force-reinstall", "--no-deps")
    print(f"Installed xgboost=={xgb_version} override for this Colab session.")

print("\nInstall complete. If you saw dependency-resolution conflicts above, "
      "restart the runtime now and re-run from the top before continuing.")


## 5. Phase 0 verification — vLLM 4-bit load + generation, T4 latency

This loads the model directly via vLLM's Python API in this notebook (not
`serving/vllm_server.py`, which is the real FastAPI deployment path for
later phases — kept as-is, unused here).


In [ ]:
import time
from vllm import LLM, SamplingParams

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    dtype="bfloat16",
    max_model_len=4096,        # kept conservative for a 16GB T4
    gpu_memory_utilization=0.85,
)

prompt = "In one sentence, what is a chargeback?"
sampling_params = SamplingParams(temperature=0.7, max_tokens=48)

start = time.perf_counter()
outputs = llm.generate([prompt], sampling_params)
latency_ms = (time.perf_counter() - start) * 1000

completion = outputs[0].outputs[0]
print("Completion:", completion.text)
print(f"\nTokens generated: {len(completion.token_ids)}")
print(f"Latency: {latency_ms:.1f} ms")
print(
    "\n*** T4 FUNCTIONAL VERIFICATION ONLY. This is NOT a representative "
    "inline-path latency number. T4 is a weak Turing-class card; the real "
    "sub-100ms inline latency claim must be re-measured on an L4 or A100 "
    "class GPU in Phase 4. ***"
)


## 6. Phase 1 — tabular model + fusion + calibration (CPU work)

Generates synthetic transaction data, trains XGBoost, fuses (currently a
tabular-score passthrough — semantic features are not wired in until Phase
2-3), calibrates, and saves a reliability diagram. All Phase 1 accuracy
numbers are directional: the data is synthetic and the point is to prove
the pipeline runs end to end, not to claim real fraud-detection accuracy.


In [ ]:
subprocess.run([sys.executable, "tests/test_phase1_smoke.py"], check=True)


In [ ]:
from IPython.display import Image, display
display(Image(filename=os.path.join(ARTIFACTS_DIR, "reliability_diagram.png")))


## 7. Commit work back to GitHub

So nothing is lost on disconnect. Requires a GitHub personal access token
stored as a Colab secret named `GITHUB_TOKEN` (key icon in the left
sidebar — grant it `repo` scope, then enable it for this notebook).


In [ ]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")
push_url = f"https://{token}@github.com/tejasp0008/Riskbrain.git"
clean_url = "https://github.com/tejasp0008/Riskbrain.git"

subprocess.run(["git", "config", "user.email", "tejas01pl@gmail.com"], check=True)
subprocess.run(["git", "config", "user.name", "tejasp0008"], check=True)
subprocess.run(["git", "remote", "set-url", "origin", push_url], check=True)

subprocess.run(["git", "add", "-A"], check=True)
status = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True, check=True)
if status.stdout.strip():
    subprocess.run(
        ["git", "commit", "-m", "Colab run: Phase 0 verification + Phase 1 artifacts"],
        check=True,
    )
    subprocess.run(["git", "push", "origin", "main"], check=True)
else:
    print("Nothing to commit, working tree is clean.")

# Reset the remote URL so the token isn't left sitting in .git/config
subprocess.run(["git", "remote", "set-url", "origin", clean_url], check=True)
print("Remote URL reset to the token-free form.")
